In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "models/Llama-3.2-3B-Instruct"

device = "cpu"
prompt = "KV cache가 뭐야?"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32,
).to(device)

model.eval()

/Users/hongkwon/miniconda3/envs/llama310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.39s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072

In [2]:
type(model)

transformers.models.llama.modeling_llama.LlamaForCausalLM

In [3]:
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs.get("attention_mask", None)

if attention_mask is not None:
    attention_mask = attention_mask.to(device)

In [4]:
with torch.no_grad():
    print("___________normal model computation___________")
    ref_outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )
    ref_logits = ref_outputs.logits

___________normal model computation___________


In [5]:
torch.set_grad_enabled(False)

### layer computation

In [6]:
hidden_states = model.model.embed_tokens(input_ids)

In [7]:
print(hidden_states.shape)

torch.Size([1, 7, 3072])


In [8]:
#________position id calculation using attention mask________
if attention_mask is not None:
    position_ids = attention_mask.cumsum(dim=-1) - 1
    position_ids.masked_fill_(attention_mask == 0, 0)
else:
    seq_len = input_ids.shape[1]
    position_ids = torch.arange(seq_len, device=device).unsqueeze(0)

In [9]:
# 중요: 최신 Llama는 RoPE를 여기서 미리 계산해서 layer에 넘김
position_embeddings = model.model.rotary_emb(
    hidden_states,
    position_ids,
)

In [10]:
position_ids.shape, position_embeddings[0].shape

(torch.Size([1, 7]), torch.Size([1, 7, 128]))

In [11]:
from transformers.cache_utils import DynamicCache
layer = next(iter(model.model.layers))
cache_position = torch.arange(7, device=model.device)
past_key_value = DynamicCache()
length = 7
out = layer(
    torch.randn([1, length, 3072]),
    attention_mask=torch.ones([1, length]), 
    position_ids= torch.randn([1, length]), 
    past_key_values=past_key_value,
    cache_position=cache_position,
    position_embeddings= (torch.randn([1, length, 128]), torch.randn([1, length, 128])),
    use_cache = True
)

In [12]:
past_key_value.layers[0].__dict__['keys'].shape

torch.Size([1, 8, 7, 128])

In [13]:
out

tensor([[[ 0.1848, -1.0834, -0.7367,  ..., -0.3892,  0.1620, -0.3258],
         [ 0.3694, -0.9250,  1.0319,  ...,  0.1328, -1.0458, -0.0145],
         [ 0.0110, -0.6645, -0.2412,  ..., -0.1814,  0.5593, -1.1777],
         ...,
         [ 1.0800, -1.0891, -0.3039,  ...,  0.5894,  0.9768, -0.4829],
         [-0.5920,  2.4760, -0.0699,  ...,  0.3235,  0.8268, -0.9857],
         [-1.1811, -0.6707,  0.7311,  ..., -2.1456, -2.3999,  0.2937]]])

In [14]:

for layer in model.model.layers:
    hidden_states = layer(
        hidden_states,
        attention_mask=None,
        position_ids=position_ids,
        position_embeddings=position_embeddings,
        use_cache=False,
    )
layers_out = hidden_states

In [15]:
layers_out.shape

torch.Size([1, 7, 3072])

In [16]:


layers_out = model.model.norm(layers_out)
output = model.lm_head(layers_out)

diff = (ref_logits - output).abs()

print("max diff :", diff.max().item())
print("mean diff:", diff.mean().item())

ref_next = ref_logits[:, -1].argmax(dim=-1)
manual_next = output[:, -1].argmax(dim=-1)

print("ref next token id   :", ref_next.item())
print("manual next token id:", manual_next.item())

print("ref next token   :", tokenizer.decode(ref_next))
print("manual next token:", tokenizer.decode(manual_next))

max diff : 0.0
mean diff: 0.0
ref next token id   : 5380
manual next token id: 5380
ref next token   : ?

manual next token: ?



In [17]:
from transformers.cache_utils import DynamicCache
import torch

layer = model.model.layers[0]
device = model.device
cache = DynamicCache()

# 1차: 0~6 위치 저장
x = torch.randn(1, 7, 3072, device=device)
position_ids = torch.arange(0, 7, device=device).unsqueeze(0)
cache_position = torch.arange(0, 7, device=device)
position_embeddings = model.model.rotary_emb(x, position_ids)

out1 = layer(
    x,
    attention_mask=None,
    position_ids=position_ids,
    past_key_values=cache,
    cache_position=cache_position,
    position_embeddings=position_embeddings,
    use_cache=True,
)

# print(cache.key_cache[0].shape)
print(cache.layers[0].__dict__['keys'].shape)
# [1, num_kv_heads, 7, head_dim]

# 2차: 7~13 위치에 새 토큰 7개 추가
# x = torch.randn(1, 7, 3072, device=device)
position_ids = torch.arange(7, 14, device=device).unsqueeze(0)
cache_position = torch.arange(7, 14, device=device)
# position_embeddings = model.model.rotary_emb(x, position_ids)

out2 = layer(
    x,
    attention_mask=None,
    position_ids=position_ids,
    past_key_values=cache,
    cache_position=cache_position,
    position_embeddings=position_embeddings,
    use_cache=True,
)
print(cache.layers[0].__dict__['keys'].shape)
# print(cache.key_cache[0].shape)
# [1, num_kv_heads, 14, head_dim]

print(f"out 1: {out1.shape}, \nout 2 : {out2.shape}")



torch.Size([1, 8, 7, 128])
torch.Size([1, 8, 14, 128])
out 1: torch.Size([1, 7, 3072]), 
out 2 : torch.Size([1, 7, 3072])


### Divide models

In [18]:
import torch
import torch.nn as nn
from transformers.cache_utils import DynamicCache
from transformers.models.llama.modeling_llama import LlamaForCausalLM


class Model1(nn.Module):
    def __init__(self, model: LlamaForCausalLM, use_cache: bool = False):
        super().__init__()

        self.use_cache = use_cache
        self.embed_tokens = model.model.embed_tokens
        self.rotary_emb = model.model.rotary_emb
        self.layers = nn.ModuleList(model.model.layers[:14])

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        past_key_values: DynamicCache | None = None,
        cache_position: torch.Tensor | None = None,
        use_cache: bool | None = None,
    ):
        if use_cache is None:
            use_cache = self.use_cache

        device = input_ids.device
        batch_size, seq_len = input_ids.shape

        hidden_states = self.embed_tokens(input_ids)

        # position_ids: [batch, seq_len]
        if attention_mask is not None:
            position_ids = attention_mask.cumsum(dim=-1) - 1
            position_ids = position_ids.masked_fill(attention_mask == 0, 0)
        else:
            position_ids = torch.arange(seq_len, device=device).unsqueeze(0)
            position_ids = position_ids.expand(batch_size, -1)

        # cache_position: [seq_len]
        if cache_position is not None:
            position_ids = cache_position.unsqueeze(0).expand(batch_size, -1)
        elif attention_mask is not None:
            position_ids = attention_mask.cumsum(dim=-1) - 1
            position_ids = position_ids.masked_fill(attention_mask == 0, 0)
        else:
            position_ids = torch.arange(seq_len, device=device).unsqueeze(0)
            position_ids = position_ids.expand(batch_size, -1)

        if use_cache and past_key_values is None:
            past_key_values = DynamicCache()

        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        for layer in self.layers:
            hidden_states = layer(
                hidden_states,
                attention_mask=None,
                position_ids=position_ids,
                position_embeddings=position_embeddings,
                past_key_values=past_key_values,
                cache_position=cache_position,
                use_cache=use_cache,
            )

        return {
            "hidden_states": hidden_states,
            "past_key_values": past_key_values,
        }

In [19]:
import torch
import torch.nn as nn
from transformers.cache_utils import DynamicCache
from transformers.models.llama.modeling_llama import LlamaForCausalLM


class Model2(nn.Module):
    def __init__(self, model: LlamaForCausalLM, use_cache: bool = False):
        super().__init__()

        self.use_cache = use_cache
        self.rotary_emb = model.model.rotary_emb
        self.layers = nn.ModuleList(model.model.layers[14:])
        self.norm = model.model.norm
        self.lm_head = model.lm_head

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        past_key_values: DynamicCache | None = None,
        cache_position: torch.Tensor | None = None,
        use_cache: bool | None = None,
    ):
        if use_cache is None:
            use_cache = self.use_cache

        device = hidden_states.device
        batch_size, seq_len, _ = hidden_states.shape

        # position_ids: [batch, seq_len]
        if attention_mask is not None:
            position_ids = attention_mask.cumsum(dim=-1) - 1
            position_ids = position_ids.masked_fill(attention_mask == 0, 0)
        else:
            position_ids = torch.arange(seq_len, device=device).unsqueeze(0)
            position_ids = position_ids.expand(batch_size, -1)
        
        if cache_position is not None:
            position_ids = cache_position.unsqueeze(0).expand(batch_size, -1)
        elif attention_mask is not None:
            position_ids = attention_mask.cumsum(dim=-1) - 1
            position_ids = position_ids.masked_fill(attention_mask == 0, 0)
        else:
            position_ids = torch.arange(seq_len, device=device).unsqueeze(0)
            position_ids = position_ids.expand(batch_size, -1)

        if use_cache and past_key_values is None:
            past_key_values = DynamicCache()

        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        for layer in self.layers:
            hidden_states = layer(
                hidden_states,
                attention_mask=None,
                position_ids=position_ids,
                position_embeddings=position_embeddings,
                past_key_values=past_key_values,
                cache_position=cache_position,
                use_cache=use_cache,
            )

        hidden_states = self.norm(hidden_states)
        logits = self.lm_head(hidden_states)

        return {
            "logits": logits,
            "hidden_states": hidden_states,
            "past_key_values": past_key_values,
        }

In [20]:
model1 = Model1(model, use_cache=True)
model2 = Model2(model, use_cache=True)

out1 = model1(input_ids, attention_mask=attention_mask)
out2 = model2(out1["hidden_states"], attention_mask=attention_mask)

logits = out2["logits"]
next_token = logits[:, -1].argmax(dim=-1, keepdim=True)

In [21]:
model.eval()
model1.eval()
model2.eval()



with torch.no_grad():
    # 1. 원본 모델
    ref = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )
    ref_logits = ref.logits

    # 2. 쪼갠 모델
    out1 = model1(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

    out2 = model2(
        hidden_states=out1["hidden_states"],
        attention_mask=attention_mask,
        use_cache=False,
    )

    pipe_logits = out2["logits"]

    # 3. 비교
    diff = (ref_logits - pipe_logits).abs()

    print("ref logits shape :", ref_logits.shape)
    print("pipe logits shape:", pipe_logits.shape)

    print("max diff :", diff.max().item())
    print("mean diff:", diff.mean().item())

    ref_next = ref_logits[:, -1].argmax(dim=-1, keepdim=True)
    pipe_next = pipe_logits[:, -1].argmax(dim=-1, keepdim=True)

    print("ref next token id :", ref_next.item())
    print("pipe next token id:", pipe_next.item())

    print("ref next token :", tokenizer.decode(ref_next[0]))
    print("pipe next token:", tokenizer.decode(pipe_next[0]))

ref logits shape : torch.Size([1, 7, 128256])
pipe logits shape: torch.Size([1, 7, 128256])
max diff : 0.0
mean diff: 0.0
ref next token id : 5380
pipe next token id: 5380
ref next token : ?

pipe next token: ?



In [22]:
model.eval()
model1.eval()
model2.eval()

generated_ref = input_ids.clone()
generated_pipe = input_ids.clone()

past_ref = None
past1 = None
past2 = None

cache_len = input_ids.shape[1]

with torch.no_grad():
    for step in range(10):
        if step == 0:
            ref_input = input_ids
            pipe_input = input_ids
            cache_position = torch.arange(input_ids.shape[1], device=input_ids.device)
        else:
            ref_input = ref_next
            pipe_input = pipe_next
            cache_position = torch.tensor([cache_len], device=input_ids.device)

        # 원본도 cache decode
        ref_out = model(
            input_ids=ref_input,
            past_key_values=past_ref,
            cache_position=cache_position,
            use_cache=True,
        )
        past_ref = ref_out.past_key_values
        ref_next = ref_out.logits[:, -1].argmax(dim=-1, keepdim=True)

        # pipeline cache decode
        out1 = model1(
            input_ids=pipe_input,
            past_key_values=past1,
            cache_position=cache_position,
            use_cache=True,
        )
        past1 = out1["past_key_values"]

        out2 = model2(
            hidden_states=out1["hidden_states"],
            past_key_values=past2,
            cache_position=cache_position,
            use_cache=True,
        )
        past2 = out2["past_key_values"]

        pipe_next = out2["logits"][:, -1].argmax(dim=-1, keepdim=True)

        generated_ref = torch.cat([generated_ref, ref_next], dim=-1)
        generated_pipe = torch.cat([generated_pipe, pipe_next], dim=-1)

        print(f"\nSTEP {step}")
        print("ref token :", ref_next.item(), tokenizer.decode(ref_next[0]))
        print("pipe token:", pipe_next.item(), tokenizer.decode(pipe_next[0]))
        print("same:", torch.equal(ref_next, pipe_next))

        cache_len += 1


STEP 0
ref token : 5380 ?

pipe token: 5380 ?

same: True

STEP 1
ref token : 83807 KV
pipe token: 83807 KV
same: True

STEP 2
ref token : 6636  cache
pipe token: 6636  cache
same: True

STEP 3
ref token : 16969 는
pipe token: 16969 는
same: True

STEP 4
ref token : 5422  Key
pipe token: 5422  Key
same: True

STEP 5
ref token : 12 -
pipe token: 12 -
same: True

STEP 6
ref token : 1150 Value
pipe token: 1150 Value
same: True

STEP 7
ref token : 20044  Cache
pipe token: 20044  Cache
same: True

STEP 8
ref token : 21028 의
pipe token: 21028 의
same: True

STEP 9
ref token : 106943  약
pipe token: 106943  약
same: True


In [23]:
test = DynamicCache()
test.__dir__()

['layers',
 'layer_class_to_replicate',
 'offloading',
 '__module__',
 '__doc__',
 '__init__',
 'to_legacy_cache',
 'from_legacy_cache',
 '__repr__',
 'prefetch',
 'offload',
 'update',
 'early_initialization',
 'get_seq_length',
 'get_mask_sizes',
 'get_max_cache_shape',
 'reset',
 'reorder_cache',
 'crop',
 'batch_repeat_interleave',
 'batch_select_indices',
 'max_batch_size',
 'max_cache_len',
 'is_compileable',
 'is_initialized',
 'is_sliding',
 '__getitem__',
 '__iter__',
 '__len__',
 '__dict__',
 '__weakref__',
 '__new__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [24]:
torch.save(model1.state_dict(), "./model1.pt")
torch.save(model2.state_dict(), "./model2.pt")

In [25]:
# model1 = Model1(base_model)
# model1.load_state_dict(torch.load("./model1.pt"))

# model2 = Model2(base_model)
# model2.load_state_dict(torch.load("./model2.pt"))

In [26]:
model.children()

<generator object Module.children at 0x7c34b4774450>

In [37]:
type(model.state_dict())

collections.OrderedDict

In [36]:
for i in model.state_dict().items(): #모델 weight만 싸그리 다 모아둔거. 이거랑 사이즈 맞으면 무조건 복사됨.
    print(type(i))

<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
<class '